# M-κ Analysis: Equilibrium Quality Investigation

This notebook investigates the equilibrium quality in the M-κ solver, particularly at early curvature values where unbalanced forces may be large.

**Objectives:**
- Use the exact same cross-section as SCADT default
- Run M-κ analysis and inspect equilibrium quality
- Investigate strain/stress distributions at problematic points
- Understand why equilibrium may be poor at low curvatures

In [ ]:
# Import required modules
import numpy as np
import matplotlib.pyplot as plt

# Material models
from bmcs_cross_section.matmod import EC2Concrete

# Cross-section design
from bmcs_cross_section.cs_design import (
    RectangularShape,
    BarReinforcement,
    ReinforcementLayout,
    CrossSection,
    StressStrainProfile
)

# Component catalogs
from bmcs_cross_section.cs_components import SteelRebarComponent

# M-κ analysis
from bmcs_cross_section.mkappa.mkappa import MKappaAnalysis

print("✓ All imports successful")

## 1. Create Default Cross-Section (Same as SCADT)

Using the exact same default configuration:
- Shape: 300mm × 500mm rectangular
- Concrete: C30/37
- Reinforcement: 4×D20 (B500B) at 50mm from bottom

In [ ]:
# Define shape (default from SCADT)
shape = RectangularShape(b=300, h=500)
print(f"Shape: {shape.b}mm × {shape.h}mm")

# Define concrete material (C30/37)
concrete = EC2Concrete(f_ck=30)
print(f"Concrete: C{concrete.f_ck}/{concrete.f_cm}")
print(f"Design strength f_cd: {concrete.f_cd:.2f} MPa")

# Define reinforcement (4×D20 at 50mm from bottom)
steel_rebar = SteelRebarComponent(nominal_diameter=20, grade='B500B')
print(f"\nRebar: D{steel_rebar.nominal_diameter} {steel_rebar.grade}")
print(f"Yield strength: {steel_rebar.matmod.f_sy:.0f} MPa")

reinforcement = ReinforcementLayout()
c_nom = 50.0  # Cover from bottom
layer_bottom = BarReinforcement(z=c_nom, component=steel_rebar, count=4)
reinforcement.layers.append(layer_bottom)

print(f"\nBottom layer: 4 bars at z={c_nom:.1f}mm")
print(f"Total steel area: {reinforcement.total_area:.0f} mm²")

# Assemble cross-section
cs = CrossSection(
    shape=shape,
    concrete=concrete,
    reinforcement=reinforcement
)

print(f"\nCross-section:")
print(f"  Total height: {cs.h_total:.0f} mm")
print(f"  Reinforcement ratio: {cs.reinforcement.total_area / cs.shape.A * 100:.2f}%")

## 2. Run M-κ Analysis with Default Parameters

In [ ]:
# Run M-κ analysis with default parameters
mkappa = MKappaAnalysis(
    cs=cs,
    n_kappa=100,
    N_tol=0.01  # 0.01 kN = 10 N tolerance
)

mkappa.solve()

print(f"M-κ Analysis complete")
print(f"  Points computed: {len(mkappa.kappa_values)}")
print(f"  Max moment: {mkappa.M_values.max():.2f} kNm")
print(f"  Max curvature: {mkappa.kappa_values[-1]*1000:.4f} ‰/mm")

## 3. Inspect Equilibrium Quality at All Points

In [ ]:
# Show detailed statistics for first 10 points
print("First 10 points of M-κ analysis:")
print("="*90)
print(f"{'#':>3} | {'κ [‰/mm]':>12} | {'M [kNm]':>10} | {'N [kN]':>10} | {'|N| [kN]':>10} | Status")
print("-"*90)

for i in range(min(10, len(mkappa.kappa_values))):
    kappa = mkappa.kappa_values[i]
    M = mkappa.M_values[i]
    N = mkappa.N_values[i]
    N_abs = abs(N)
    
    # Status indicator
    if N_abs < 0.01:
        status = "✓ Good"
    elif N_abs < 0.1:
        status = "⚠ Acceptable"
    elif N_abs < 1.0:
        status = "⚠ Poor"
    else:
        status = "✗ Failed"
    
    print(f"{i:3d} | {kappa*1000:12.6f} | {M:10.3f} | {N:10.3f} | {N_abs:10.3f} | {status}")

print("="*90)
print(f"\nOverall statistics:")
print(f"  Mean |N|: {np.abs(mkappa.N_values).mean():.4f} kN")
print(f"  Max |N|: {np.abs(mkappa.N_values).max():.4f} kN")
print(f"  Median |N|: {np.median(np.abs(mkappa.N_values)):.4f} kN")
print(f"  Points with |N| > 0.1 kN: {np.sum(np.abs(mkappa.N_values) > 0.1)}")
print(f"  Points with |N| > 1.0 kN: {np.sum(np.abs(mkappa.N_values) > 1.0)}")

## 4. Visualize Equilibrium Quality

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: M-κ curve
ax1 = axes[0, 0]
ax1.plot(mkappa.kappa_values*1000, mkappa.M_values, 'b-', linewidth=2)
ax1.set_xlabel('Curvature [1/m]', fontsize=11)
ax1.set_ylabel('Moment M [kNm]', fontsize=11)
ax1.set_title('Moment-Curvature Relationship', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Plot 2: N vs κ (equilibrium error)
ax2 = axes[0, 1]
ax2.plot(mkappa.kappa_values*1000, mkappa.N_values, 'r-', linewidth=2, label='N (axial force)')
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax2.axhline(y=0.01, color='g', linestyle='--', alpha=0.5, label='Tolerance (±0.01 kN)')
ax2.axhline(y=-0.01, color='g', linestyle='--', alpha=0.5)
ax2.set_xlabel('Curvature [1/m]', fontsize=11)
ax2.set_ylabel('Axial Force N [kN]', fontsize=11)
ax2.set_title('Equilibrium Error (should be ≈0)', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: |N| vs κ (log scale)
ax3 = axes[1, 0]
ax3.semilogy(mkappa.kappa_values*1000, np.abs(mkappa.N_values), 'r-', linewidth=2)
ax3.axhline(y=0.01, color='g', linestyle='--', alpha=0.5, label='Tolerance (0.01 kN)')
ax3.axhline(y=0.1, color='orange', linestyle='--', alpha=0.5, label='Poor (0.1 kN)')
ax3.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Failed (1.0 kN)')
ax3.set_xlabel('Curvature [1/m]', fontsize=11)
ax3.set_ylabel('|Axial Force| |N| [kN]', fontsize=11)
ax3.set_title('Absolute Equilibrium Error (Log Scale)', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3, which='both')

# Plot 4: |N| for first 20 points (zoomed)
ax4 = axes[1, 1]
n_zoom = min(20, len(mkappa.kappa_values))
indices = np.arange(n_zoom)
ax4.bar(indices, np.abs(mkappa.N_values[:n_zoom]), color='red', alpha=0.6)
ax4.axhline(y=0.01, color='g', linestyle='--', linewidth=2, label='Tolerance (0.01 kN)')
ax4.axhline(y=0.1, color='orange', linestyle='--', linewidth=2, label='Poor (0.1 kN)')
ax4.set_xlabel('Point Index', fontsize=11)
ax4.set_ylabel('|N| [kN]', fontsize=11)
ax4.set_title('First 20 Points: Equilibrium Error', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Investigate Strain/Stress at Problematic Points

Let's examine the first few points in detail to understand what's happening.

In [ ]:
# Analyze first 3 points in detail
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for i in range(min(3, len(mkappa.kappa_values))):
    kappa = mkappa.kappa_values[i]
    eps_bottom = mkappa.eps_bot_values[i]
    M = mkappa.M_values[i]
    N = mkappa.N_values[i]
    
    # Create stress-strain profile
    profile = StressStrainProfile(cs, kappa, eps_bottom)
    
    # Get strain and stress profiles
    z_strain, strains = profile.get_strain_profile(n_points=100)
    z_stress, stresses = profile.get_concrete_stress_profile(n_points=100)
    z_reinf, eps_reinf, sig_reinf = profile.get_reinforcement_strains_stresses()
    
    # Get force resultants
    F_c, F_s, N_total, M_total = profile.get_force_resultants()
    
    # Plot strain profile
    ax_strain = axes[i, 0]
    ax_strain.plot(strains * 1000, z_strain, 'b-', linewidth=2, label='Strain profile')
    ax_strain.axvline(x=0, color='k', linestyle='--', alpha=0.5)
    
    # Mark reinforcement
    for z, eps in zip(z_reinf, eps_reinf):
        # Convert to scalar properly
        eps_val = eps.item() if isinstance(eps, np.ndarray) else float(eps)
        z_val = z.item() if isinstance(z, np.ndarray) else float(z)
        ax_strain.plot(eps * 1000, z, 'ro', markersize=10)
        ax_strain.text(eps_val * 1000 + 0.1, z_val, f'{eps_val:.4f}', fontsize=9, va='center')
    
    ax_strain.set_xlabel('Strain [‰]', fontsize=11)
    ax_strain.set_ylabel('Height z [mm]', fontsize=11)
    ax_strain.set_title(f'Point {i}: Strain Distribution\nκ={kappa*1000:.6f} ‰/mm', 
                       fontsize=12, fontweight='bold')
    ax_strain.grid(True, alpha=0.3)
    ax_strain.set_ylim(0, cs.h_total)
    
    # Plot stress profile
    ax_stress = axes[i, 1]
    ax_stress.plot(stresses, z_stress, 'b-', linewidth=2, label='Concrete stress')
    ax_stress.fill_betweenx(z_stress, 0, stresses, alpha=0.2, color='blue')
    ax_stress.axvline(x=0, color='k', linestyle='--', alpha=0.5)
    
    # Mark reinforcement
    for z, sig in zip(z_reinf, sig_reinf):
        # Convert to scalar properly
        sig_val = sig.item() if isinstance(sig, np.ndarray) else float(sig)
        z_val = z.item() if isinstance(z, np.ndarray) else float(z)
        color = 'red' if sig_val > 0 else 'darkred'
        ax_stress.plot(sig, z, 'o', markersize=12, color=color, 
                      markeredgecolor='black', markeredgewidth=2)
        ax_stress.text(sig_val + 2, z_val, f'{sig_val:.1f} MPa', fontsize=9, va='center')
    
    ax_stress.set_xlabel('Stress σ [MPa]', fontsize=11)
    ax_stress.set_ylabel('Height z [mm]', fontsize=11)
    ax_stress.set_title(f'Point {i}: Stress Distribution\nN={N:.3f} kN, M={M:.3f} kNm', 
                       fontsize=12, fontweight='bold')
    ax_stress.grid(True, alpha=0.3)
    ax_stress.set_ylim(0, cs.h_total)
    
    # Add force info as text
    info_text = f'F_c={F_c/1000:.3f} kN\nF_s={F_s/1000:.3f} kN\nSum={N_total/1000:.3f} kN'
    ax_stress.text(0.02, 0.98, info_text, transform=ax_stress.transAxes,
                  fontsize=10, verticalalignment='top',
                  bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout(pad=3.0)
plt.show()


## 6. Analysis: Why Large N at Low Curvatures?

Let's investigate the solver behavior at very small curvatures.

In [ ]:
# Check the initial curvature and strain values
print("Analysis of First Point:")
print("="*60)
kappa_0 = mkappa.kappa_values[0]
eps_bottom_0 = mkappa.eps_bot_values[0]
N_0 = mkappa.N_values[0]
M_0 = mkappa.M_values[0]

print(f"κ = {kappa_0*1000:.8f} ‰/mm (= {kappa_0*1e6:.4f} × 10⁻⁶ mm⁻¹)")
print(f"ε_bottom = {eps_bottom_0:.8f}")
print(f"N = {N_0:.6f} kN")
print(f"M = {M_0:.6f} kNm")

# Calculate strain at reinforcement level
eps_steel = eps_bottom_0 + kappa_0 * c_nom
print(f"\nε at steel (z={c_nom}mm) = {eps_steel:.8f}")

# Check if steel is in elastic range
E_s = steel_rebar.matmod.E_s
f_sy = steel_rebar.matmod.f_sy
eps_y = f_sy / E_s
print(f"\nSteel yield strain ε_y = {eps_y:.6f}")
print(f"Steel elastic? {abs(eps_steel) < eps_y}")

# Calculate steel stress
sig_steel = steel_rebar.matmod.get_sig(eps_steel)
print(f"Steel stress σ_s = {sig_steel:.2f} MPa")
print(f"Steel force F_s = {sig_steel * reinforcement.total_area / 1000:.3f} kN")

# Check concrete strains
eps_top = eps_bottom_0 + kappa_0 * cs.h_total
eps_mid = eps_bottom_0 + kappa_0 * cs.h_total / 2
print(f"\nε at top (z={cs.h_total}mm) = {eps_top:.8f}")
print(f"\nε at mid (z={cs.h_total/2}mm) = {eps_mid:.8f}")

# Is section mostly in compression or tension?
if eps_bottom_0 > 0 and eps_top > 0:
    print("\n⚠ Section is entirely in TENSION")
elif eps_bottom_0 < 0 and eps_top < 0:
    print("\n⚠ Section is entirely in COMPRESSION")
else:
    neutral_axis = -eps_bottom_0 / kappa_0
    print(f"\n✓ Neutral axis at z = {neutral_axis:.2f} mm")
    print(f"  Distance from bottom: {neutral_axis:.2f} mm")
    print(f"  Distance from top: {cs.h_total - neutral_axis:.2f} mm")


## 7. Test: Manual Equilibrium Calculation

Let's manually calculate forces for the first point to verify the solver.

In [ ]:
# Manual verification for point 0
kappa = mkappa.kappa_values[0]
eps_bottom = mkappa.eps_bot_values[0]

profile = StressStrainProfile(cs, kappa, eps_bottom)

# Get detailed force breakdown
F_c, F_s, N_total, M_total = profile.get_force_resultants()

print("Manual Equilibrium Verification:")
print("="*60)
print(f"Concrete force F_c = {F_c/1000:.6f} kN")
print(f"Steel force F_s = {F_s/1000:.6f} kN")
print(f"Total axial force N = {N_total/1000:.6f} kN")
print(f"\nCheck: F_c + F_s = {(F_c + F_s)/1000:.6f} kN")
print(f"Error: {abs(F_c + F_s - N_total):.6f} N (should be ≈0)")

# Check if this matches what's stored
print(f"\nStored N value: {mkappa.N_values[0]:.6f} kN")
print(f"Match? {abs(mkappa.N_values[0] - N_total/1000) < 1e-6}")

# Analyze concrete contribution
z_coords = np.linspace(0, cs.h_total, 100)
strains = np.array([profile.get_strain_at_z(z) for z in z_coords])
stresses = np.array([cs.concrete.get_sig(eps) for eps in strains])

# Integrate manually
dz = z_coords[1] - z_coords[0]
F_c_manual = np.sum(stresses * cs.shape.b * dz)
print(f"\nManual concrete force integration: {F_c_manual/1000:.6f} kN")
print(f"Matches profile.get_force_resultants()? {abs(F_c_manual - F_c) < 1}")


## 8. Investigation: Sensitivity to Initial Conditions

In [ ]:
# Test different numbers of points to see if convergence improves
n_points_list = [50, 100, 200]
results = []

for n_points in n_points_list:
    mk = MKappaAnalysis(cs=cs, n_kappa=n_points, N_tol=0.01)
    mk.solve()
    
    # Check first 5 points
    first_5_N = np.abs(mk.N_values[:min(5, len(mk.N_values))])
    
    results.append({
        'n_points': n_points,
        'total_points': len(mk.kappa_values),
        'first_5_mean_N': first_5_N.mean(),
        'first_5_max_N': first_5_N.max(),
        'overall_mean_N': np.abs(mk.N_values).mean(),
        'overall_max_N': np.abs(mk.N_values).max()
    })

print("Sensitivity to Number of Points:")
print("="*90)
print(f"{'n_points':<10} | {'Points':<8} | {'First 5 Mean':<15} | {'First 5 Max':<15} | {'Overall Mean':<15}")
print("-"*90)
for r in results:
    print(f"{r['n_points']:<10} | {r['total_points']:<8} | {r['first_5_mean_N']:<15.4f} | {r['first_5_max_N']:<15.4f} | {r['overall_mean_N']:<15.4f}")

## 9. Summary and Recommendations

Based on this investigation, we can identify:

1. **Root Cause**: The equilibrium quality at low curvatures
2. **Impact**: How it affects the M-κ curve
3. **Solutions**: Potential improvements to the solver

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("INVESTIGATION SUMMARY")
print("="*70)

# Find which points have poor equilibrium
poor_equilibrium = np.abs(mkappa.N_values) > 0.1
n_poor = np.sum(poor_equilibrium)
pct_poor = (n_poor / len(mkappa.N_values)) * 100

print(f"\n1. Equilibrium Quality:")
print(f"   - Points with |N| > 0.1 kN: {n_poor} ({pct_poor:.1f}%)")
print(f"   - Worst equilibrium error: {np.abs(mkappa.N_values).max():.4f} kN")
print(f"   - Mean equilibrium error: {np.abs(mkappa.N_values).mean():.4f} kN")

if n_poor > 0:
    poor_indices = np.where(poor_equilibrium)[0]
    print(f"   - Poor points occur at indices: {poor_indices[:10].tolist()}...")
    print(f"   - Concentrated at beginning? {np.all(poor_indices[:5] < 10)}")

print(f"\n2. Curvature Range:")
print(f"   - κ_min = {mkappa.kappa_values[0]*1000:.6f} ‰/mm")
print(f"   - κ_max = {mkappa.kappa_values[-1]*1000:.6f} ‰/mm")
print(f"   - κ_step (avg) = {np.diff(mkappa.kappa_values).mean()*1000:.6f} ‰/mm")

print(f"\n3. Recommendations:")
if np.abs(mkappa.N_values[0]) > 1.0:
    print(f"   ⚠ CRITICAL: Very large N at first point ({mkappa.N_values[0]:.3f} kN)")
    print(f"   → May indicate solver initialization issue")
    print(f"   → Consider starting from a different initial strain state")
elif np.all(np.abs(mkappa.N_values[:5]) > 0.1):
    print(f"   ⚠ WARNING: Poor equilibrium in first 5 points")
    print(f"   → May need better initial guess for eps_bottom")
    print(f"   → Consider adaptive step size at beginning")
else:
    print(f"   ✓ Equilibrium quality is generally acceptable")
    print(f"   → Current solver settings are adequate")

print("\n" + "="*70)